# Prepare cups dataset for ColonyNet

This notebook:
1. Reads all raw images from the cups raw root (auto-detected).
2. Builds instance masks from available annotations (`.tif`, `_colored.tif`, `*_coordinates.txt`).
3. Applies robust Petri-dish crop to image and mask (mask-guided + hough fallback) on original resolution.
4. Applies one final resize-with-padding to 512x512.
5. Saves in ColonyNet format: `images/*` + `instances/*.png` (instance ids).
6. Generates mild augmentations to reach x10 dataset size.
7. Merges result into `trainable_pool` and updates `trainable_pool/report.json`.


In [ ]:
from pathlib import Path

RAW_ROOT_NAME = '\u0434\u0430\u0442\u0430\u0441\u0435\u0442\u044b_\u0447\u0430\u0448\u043a\u0438'
MARKUP_TOKEN = '\u0440\u0430\u0437\u043c\u0435\u0442'

def resolve_raw_root() -> Path:
    direct = Path(RAW_ROOT_NAME)
    if direct.exists():
        return direct

    cwd = Path('.')
    for d in cwd.iterdir():
        if not d.is_dir():
            continue
        child_dirs = {c.name.lower() for c in d.iterdir() if c.is_dir()}
        if ('s. aureus' in child_dirs or 's. aureus 2' in child_dirs) and any('coli' in x for x in child_dirs):
            return d

    raise FileNotFoundError(f'Could not locate raw cups dataset root. Expected folder: {RAW_ROOT_NAME}')

RAW_ROOT = resolve_raw_root()
OUT_ROOT = Path('data/cups_dataset_all_petri_aug10_512')
POOL_ROOT = Path('trainable_pool')

POOL_PREFIX = 'data_cups_dataset_all_petri_aug10'
REPORT_DATASET_KEY = 'data/cups_dataset_all_petri_aug10_512'

AUG_MULTIPLIER = 10  # final count = originals * 10
SAFE_GEOM_AUG = True
SEED = 42
INCLUDE_UNLABELED_AS_EMPTY = True

USE_MASK_GUIDED_CROP = True
MASK_OUTSIDE_CIRCLE = True
PAD_FRAC = 0.08

# Hough/fallback circle bounds
MIN_R_FRAC = 0.18
MAX_R_FRAC = 0.80
CENTER_TOL = 0.45

# Guard against over/under crop: crop area / original area
CROP_AREA_MIN_RATIO = 0.15
CROP_AREA_MAX_RATIO = 1.00
CENTER_FALLBACK_R_FRAC = 0.39  # used when detector fails

# Optional: set integer for quick debug, keep None for full run
MAX_FILES = None
TARGET_SIZE = 512

RAW_ROOT, OUT_ROOT, POOL_ROOT


In [ ]:
import json
import re
import shutil
from collections import defaultdict
from datetime import datetime, timezone

import cv2
import numpy as np

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

import matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries

def detect_petri_circle(img_rgb, min_r_frac=0.35, max_r_frac=0.55, center_tol=0.25):
    h, w = img_rgb.shape[:2]
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (9, 9), 2)

    min_r = int(min(h, w) * min_r_frac)
    max_r = int(min(h, w) * max_r_frac)

    circles = cv2.HoughCircles(
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=min(h, w)//2,
        param1=100, param2=30, minRadius=min_r, maxRadius=max_r
    )
    if circles is not None:
        circles = np.round(circles[0]).astype(int)
        cx, cy, r = circles[np.argmax(circles[:,2])]
    else:
        edges = cv2.Canny(gray, 50, 150)
        cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            return None
        cnt = max(cnts, key=cv2.contourArea)
        (cx_f, cy_f), r_f = cv2.minEnclosingCircle(cnt)
        cx, cy, r = int(cx_f), int(cy_f), int(r_f)

    if r < min_r or r > max_r:
        return None
    cx0, cy0 = w // 2, h // 2
    max_off = center_tol * min(h, w)
    if ((cx - cx0)**2 + (cy - cy0)**2) ** 0.5 > max_off:
        return None
    return cx, cy, r

def crop_petri(img_rgb, pad=0.02, mask_outside=True, min_r_frac=0.35, max_r_frac=0.55, center_tol=0.25):
    h, w = img_rgb.shape[:2]
    circ = detect_petri_circle(img_rgb, min_r_frac=min_r_frac, max_r_frac=max_r_frac, center_tol=center_tol)
    if circ is None:
        return img_rgb, None, 'no_circle'
    cx, cy, r = circ
    r = int(r * (1.0 + pad))

    x1, y1 = max(0, cx - r), max(0, cy - r)
    x2, y2 = min(w, cx + r), min(h, cy + r)

    crop = img_rgb[y1:y2, x1:x2].copy()

    if mask_outside:
        yy, xx = np.ogrid[y1:y2, x1:x2]
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= (r * r)
        crop[~mask] = 0

    return crop, (cx, cy, r, x1, y1, x2, y2), 'cropped_hough'

def overlay_boundaries(img, lbl):
    out = img.copy()
    b = find_boundaries(lbl, mode='outer')
    out[b] = (0, 255, 0)
    return out

IMG_ID_RE = re.compile(r'(IMG_\d+)', flags=re.IGNORECASE)
COORD_RE = re.compile(r'\((\d+),\s*(\d+)\)')
RAW_IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
MASK_TIF_EXTS = {'.tif', '.tiff'}

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def reset_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def read_image_any(path: Path, flags: int) -> np.ndarray | None:
    buf = np.fromfile(str(path), dtype=np.uint8)
    if buf.size == 0:
        return None
    return cv2.imdecode(buf, flags)

def write_image_rgb(path: Path, img_rgb: np.ndarray) -> None:
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    ext = path.suffix.lower()
    params = []
    if ext in {'.jpg', '.jpeg'}:
        params = [int(cv2.IMWRITE_JPEG_QUALITY), 95]
    ok, enc = cv2.imencode(ext, bgr, params)
    if not ok:
        raise RuntimeError(f'Failed to encode image: {path}')
    enc.tofile(str(path))

def write_png_mask(path: Path, mask: np.ndarray) -> None:
    ok, enc = cv2.imencode('.png', mask)
    if not ok:
        raise RuntimeError(f'Failed to encode mask: {path}')
    enc.tofile(str(path))


def resize_with_padding(arr: np.ndarray, size: int, is_mask: bool) -> np.ndarray:
    h, w = arr.shape[:2]
    if h <= 0 or w <= 0:
        raise ValueError(f'Invalid image shape: {arr.shape}')

    scale = min(size / float(h), size / float(w))
    new_h = max(1, int(round(h * scale)))
    new_w = max(1, int(round(w * scale)))

    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_AREA
    resized = cv2.resize(arr, (new_w, new_h), interpolation=interp)

    pad_top = (size - new_h) // 2
    pad_bottom = size - new_h - pad_top
    pad_left = (size - new_w) // 2
    pad_right = size - new_w - pad_left

    if resized.ndim == 3:
        padded = cv2.copyMakeBorder(resized, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    else:
        padded = cv2.copyMakeBorder(resized, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_CONSTANT, value=0)

    if padded.shape[0] != size or padded.shape[1] != size:
        padded = cv2.resize(padded, (size, size), interpolation=interp)
    return padded


def force_size_pair(img_rgb: np.ndarray, inst: np.ndarray, size: int) -> tuple[np.ndarray, np.ndarray]:
    out_img = resize_with_padding(img_rgb, size=size, is_mask=False)
    out_inst = resize_with_padding(inst, size=size, is_mask=True)
    if out_inst.dtype != np.uint16:
        out_inst = out_inst.astype(np.uint16)
    return out_img, out_inst

def to_binary(mask: np.ndarray) -> np.ndarray:
    if mask.ndim == 3:
        fg = np.any(mask > 0, axis=2)
    else:
        fg = mask > 0
    return fg.astype(np.uint8)

def binary_to_instances(binary: np.ndarray) -> np.ndarray:
    _, labels = cv2.connectedComponents(binary, connectivity=8)
    max_id = int(labels.max())
    if max_id > np.iinfo(np.uint16).max:
        raise RuntimeError(f'Too many instances ({max_id}); exceeds uint16.')
    return labels.astype(np.uint16)

def parse_coordinates_to_binary(txt_path: Path, h: int, w: int) -> np.ndarray:
    mask = np.zeros((h, w), dtype=np.uint8)
    with txt_path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = COORD_RE.search(line)
            if m is None:
                continue
            x = int(m.group(1))
            y = int(m.group(2))
            if 0 <= x < w and 0 <= y < h:
                mask[y, x] = 1
    return mask

def sanitize_id(text: str) -> str:
    out = re.sub(r'[^A-Za-z0-9_]+', '_', text).strip('_')
    return out or 'sample'

def unique_id(base_id: str, used: set[str]) -> str:
    if base_id not in used:
        used.add(base_id)
        return base_id
    i = 2
    while True:
        cand = f'{base_id}__{i}'
        if cand not in used:
            used.add(cand)
            return cand
        i += 1

def common_prefix_len(a_parts: tuple[str, ...], b_parts: tuple[str, ...]) -> int:
    n = 0
    for x, y in zip(a_parts, b_parts):
        if x.casefold() != y.casefold():
            break
        n += 1
    return n

def choose_closest_path(img_path: Path, candidates: list[Path]) -> Path:
    img_parts = tuple(str(p) for p in img_path.resolve().parts)
    ranked = sorted(candidates, key=lambda p: (-common_prefix_len(img_parts, tuple(str(x) for x in p.resolve().parts)), len(str(p))))
    return ranked[0]

def collect_raw_images(raw_root: Path) -> list[Path]:
    imgs = []
    for p in raw_root.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in RAW_IMG_EXTS:
            continue
        parts_cf = [x.casefold() for x in p.parts]
        if any(MARKUP_TOKEN in part for part in parts_cf):
            continue
        name_l = p.name.lower()
        if 'colored' in name_l or 'marked' in name_l:
            continue
        imgs.append(p)
    return sorted(imgs)

def collect_annotations(raw_root: Path):
    base_by_stem = defaultdict(list)
    marked_by_stem = defaultdict(list)
    colored_by_stem = defaultdict(list)
    coords_by_stem = defaultdict(list)
    for p in raw_root.rglob('*'):
        if not p.is_file():
            continue
        ext = p.suffix.lower()
        name_l = p.name.lower()
        if ext in MASK_TIF_EXTS:
            m = IMG_ID_RE.search(p.name)
            if m is None:
                continue
            stem = m.group(1).upper()
            if 'colored' in name_l:
                colored_by_stem[stem].append(p)
            elif 'marked' in name_l:
                marked_by_stem[stem].append(p)
            else:
                base_by_stem[stem].append(p)
        elif ext == '.txt' and 'coord' in name_l:
            m = IMG_ID_RE.search(p.name)
            if m is None:
                continue
            coords_by_stem[m.group(1).upper()].append(p)
    return base_by_stem, marked_by_stem, colored_by_stem, coords_by_stem

def _crop_with_circle(img_rgb: np.ndarray, inst: np.ndarray, cx: int, cy: int, r: int, pad: float, mask_outside: bool):
    h, w = img_rgb.shape[:2]
    r = int(r * (1.0 + pad))
    x1, y1 = max(0, cx - r), max(0, cy - r)
    x2, y2 = min(w, cx + r), min(h, cy + r)
    crop_img = img_rgb[y1:y2, x1:x2].copy()
    crop_inst = inst[y1:y2, x1:x2].copy()
    if mask_outside:
        yy, xx = np.ogrid[y1:y2, x1:x2]
        inside = (xx - cx) ** 2 + (yy - cy) ** 2 <= (r * r)
        crop_img[~inside] = 0
        crop_inst[~inside] = 0
    return crop_img, crop_inst, (cx, cy, r, x1, y1, x2, y2)

def crop_from_instance_mask(img_rgb: np.ndarray, inst: np.ndarray, pad: float, mask_outside: bool):
    fg = (inst > 0).astype(np.uint8)
    if int(fg.sum()) == 0:
        return None
    pts = cv2.findNonZero(fg)
    if pts is None:
        return None
    (cx_f, cy_f), r_f = cv2.minEnclosingCircle(pts)
    cx, cy, r = int(cx_f), int(cy_f), int(r_f)
    if r < 8:
        return None
    crop_img, crop_inst, meta = _crop_with_circle(img_rgb, inst, cx, cy, r, pad, mask_outside)
    return crop_img, crop_inst, meta, 'cropped_mask'


def crop_center_fallback(img_rgb: np.ndarray, inst: np.ndarray, r_frac: float, pad: float, mask_outside: bool):
    h, w = img_rgb.shape[:2]
    cx, cy = w // 2, h // 2
    r = int(min(h, w) * float(r_frac))
    if r < 8:
        return img_rgb, inst, 'copy_too_small'
    cimg, cinst, _ = _crop_with_circle(img_rgb, inst, cx, cy, r, pad, mask_outside)
    return cimg, cinst, 'cropped_center_fallback'

def _area_ratio(crop_img: np.ndarray, orig_img: np.ndarray) -> float:
    h, w = crop_img.shape[:2]
    oh, ow = orig_img.shape[:2]
    return float((h * w) / max(1, (oh * ow)))

def crop_petri_pair(img_rgb: np.ndarray, inst: np.ndarray, pad: float, mask_outside: bool, prefer_mask_guided: bool, min_r_frac: float, max_r_frac: float, center_tol: float, area_min_ratio: float, area_max_ratio: float, center_fallback_r_frac: float):
    # 1) Try mask-guided crop for labeled samples
    if prefer_mask_guided:
        m = crop_from_instance_mask(img_rgb, inst, pad=pad, mask_outside=mask_outside)
        if m is not None:
            cimg, cinst, _meta, status = m
            ar = _area_ratio(cimg, img_rgb)
            if area_min_ratio <= ar <= area_max_ratio:
                return cimg, cinst, status

    # 2) Hough/fallback crop from image
    h_img, meta, status = crop_petri(
        img_rgb,
        pad=pad,
        mask_outside=mask_outside,
        min_r_frac=min_r_frac,
        max_r_frac=max_r_frac,
        center_tol=center_tol,
    )
    if status == 'no_circle' or meta is None:
        return crop_center_fallback(img_rgb, inst, r_frac=center_fallback_r_frac, pad=pad, mask_outside=mask_outside)

    cx, cy, r, x1, y1, x2, y2 = meta
    cinst = inst[y1:y2, x1:x2].copy()
    if mask_outside:
        yy, xx = np.ogrid[y1:y2, x1:x2]
        inside = (xx - cx) ** 2 + (yy - cy) ** 2 <= (r * r)
        cinst[~inside] = 0

    ar = _area_ratio(h_img, img_rgb)
    if not (area_min_ratio <= ar <= area_max_ratio):
        return img_rgb, inst, 'copy_area_guard'

    return h_img, cinst, 'cropped_hough'

def _random_affine_pair(img_rgb: np.ndarray, inst: np.ndarray, rng: np.random.Generator):
    out_img, out_inst = img_rgb, inst

    # Safe geometry: no translations/scaling/free-angle rotations to avoid cutting Petri dish.
    if rng.random() < 0.5:
        out_img = np.ascontiguousarray(np.fliplr(out_img))
        out_inst = np.ascontiguousarray(np.fliplr(out_inst))

    if rng.random() < 0.2:
        out_img = np.ascontiguousarray(np.flipud(out_img))
        out_inst = np.ascontiguousarray(np.flipud(out_inst))

    if SAFE_GEOM_AUG and rng.random() < 0.5:
        k = int(rng.integers(1, 4))  # 90/180/270
        out_img = np.ascontiguousarray(np.rot90(out_img, k))
        out_inst = np.ascontiguousarray(np.rot90(out_inst, k))

    return out_img, out_inst

def _random_photo(img_rgb: np.ndarray, rng: np.random.Generator):
    out = img_rgb.astype(np.float32)

    # Stronger photometric diversity while staying plausible for Petri images.
    if rng.random() < 0.98:
        out = out * float(rng.uniform(0.80, 1.24)) + float(rng.uniform(-30.0, 30.0))

    if rng.random() < 0.55:
        g = float(rng.uniform(0.72, 1.28))
        out = np.power(np.clip(out / 255.0, 0.0, 1.0), g) * 255.0

    out = np.clip(out, 0, 255).astype(np.uint8)

    if rng.random() < 0.65:
        hsv = cv2.cvtColor(out, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[..., 0] = (hsv[..., 0] + float(rng.uniform(-10.0, 10.0))) % 180.0
        hsv[..., 1] *= float(rng.uniform(0.65, 1.45))
        hsv[..., 2] *= float(rng.uniform(0.80, 1.25))
        out = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB)

    if rng.random() < 0.35:
        # Channel-wise gain (white-balance drift)
        gains = rng.uniform(0.88, 1.12, size=(1, 1, 3)).astype(np.float32)
        out = np.clip(out.astype(np.float32) * gains, 0, 255).astype(np.uint8)

    if rng.random() < 0.40:
        k = int(rng.choice([3, 5]))
        out = cv2.GaussianBlur(out, (k, k), sigmaX=float(rng.uniform(0.2, 1.6)))

    if rng.random() < 0.38:
        blur = cv2.GaussianBlur(out, (0, 0), sigmaX=float(rng.uniform(0.8, 1.8)))
        out = cv2.addWeighted(out, 1.45, blur, -0.45, 0)

    if rng.random() < 0.55:
        sigma = float(rng.uniform(4.0, 14.0))
        noise = rng.normal(0.0, sigma, size=out.shape).astype(np.float32)
        out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    if rng.random() < 0.35:
        # Illumination field (vignette / gradient-like variation)
        h, w = out.shape[:2]
        yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
        cx = float(rng.uniform(0.35 * w, 0.65 * w))
        cy = float(rng.uniform(0.35 * h, 0.65 * h))
        r2 = ((xx - cx) ** 2 + (yy - cy) ** 2)
        r2 /= max(1.0, float((0.8 * max(h, w)) ** 2))
        a = float(rng.uniform(-0.20, 0.20))
        gain = np.clip(1.0 + a * r2, 0.75, 1.25)
        out = np.clip(out.astype(np.float32) * gain[..., None], 0, 255).astype(np.uint8)

    if rng.random() < 0.45:
        q = int(rng.integers(95, 101))
        ok, enc = cv2.imencode('.jpg', cv2.cvtColor(out, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), q])
        if ok:
            dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
            if dec is not None:
                out = cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)

    return out

def augment_pair(img_rgb: np.ndarray, inst: np.ndarray, rng: np.random.Generator):
    g_img, g_inst = _random_affine_pair(img_rgb, inst, rng)
    return _random_photo(g_img, rng), g_inst

def validation_stats(images_dir: Path, instances_dir: Path) -> dict[str, int]:
    image_stems = {p.stem for p in images_dir.iterdir() if p.is_file()}
    inst_stems = {p.stem for p in instances_dir.iterdir() if p.is_file() and p.suffix.lower() == '.png'}
    return {'images_total': len(image_stems), 'instances_total': len(inst_stems), 'matched_pairs': len(image_stems & inst_stems), 'only_images': len(image_stems - inst_stems), 'only_instances': len(inst_stems - image_stems)}


In [ ]:
def build_dataset(raw_root: Path, out_root: Path) -> dict:
    out_images = out_root / 'images'
    out_instances = out_root / 'instances'
    reset_dir(out_images); reset_dir(out_instances)

    raw_images = collect_raw_images(raw_root)
    if MAX_FILES is not None:
        raw_images = raw_images[: int(MAX_FILES)]

    base_by_stem, marked_by_stem, colored_by_stem, coords_by_stem = collect_annotations(raw_root)

    used_ids: set[str] = set()
    rng = np.random.default_rng(SEED)

    stats = {
        'raw_images_total': len(raw_images),
        'processed_base_samples': 0,
        'saved_with_augs': 0,
        'source_kind': {'base_tif': 0, 'marked_tif': 0, 'colored_tif': 0, 'coordinates_txt': 0, 'empty': 0},
        'crop_status': {'cropped_mask': 0, 'cropped_hough': 0, 'cropped_center_fallback': 0, 'copy_area_guard': 0, 'copy_too_small': 0},
        'skipped': {'no_annotation': 0, 'image_read_fail': 0, 'mask_read_fail': 0},
    }
    skipped_examples = []

    def pick_mask(stem_up: str, img_path: Path):
        if stem_up in base_by_stem: return 'base_tif', choose_closest_path(img_path, base_by_stem[stem_up])
        if stem_up in marked_by_stem: return 'marked_tif', choose_closest_path(img_path, marked_by_stem[stem_up])
        if stem_up in colored_by_stem: return 'colored_tif', choose_closest_path(img_path, colored_by_stem[stem_up])
        if stem_up in coords_by_stem: return 'coordinates_txt', choose_closest_path(img_path, coords_by_stem[stem_up])
        return None, None

    for img_path in tqdm(raw_images, desc='convert+resize+crop+augment'):
        img_bgr = read_image_any(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            stats['skipped']['image_read_fail'] += 1
            if len(skipped_examples) < 100: skipped_examples.append(f'image_read_fail:{img_path}')
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        stem_up = img_path.stem.upper()
        source_kind, ann_path = pick_mask(stem_up, img_path)

        if source_kind is None:
            if INCLUDE_UNLABELED_AS_EMPTY:
                inst = np.zeros((h, w), dtype=np.uint16); source_kind = 'empty'
            else:
                stats['skipped']['no_annotation'] += 1
                if len(skipped_examples) < 100: skipped_examples.append(f'no_annotation:{img_path}')
                continue
        elif source_kind == 'coordinates_txt':
            inst = binary_to_instances(parse_coordinates_to_binary(ann_path, h, w))
        else:
            mask = read_image_any(ann_path, cv2.IMREAD_UNCHANGED)
            if mask is None:
                stats['skipped']['mask_read_fail'] += 1
                if len(skipped_examples) < 100: skipped_examples.append(f'mask_read_fail:{ann_path}')
                continue
            mask_bin = to_binary(mask)
            if mask_bin.shape[:2] != (h, w):
                mask_bin = (cv2.resize(mask_bin, (w, h), interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8)
            inst = binary_to_instances(mask_bin)

        crop_img, crop_inst, crop_status = crop_petri_pair(
            img_rgb=img_rgb, inst=inst,
            pad=PAD_FRAC, mask_outside=MASK_OUTSIDE_CIRCLE,
            prefer_mask_guided=(USE_MASK_GUIDED_CROP and source_kind != 'empty'),
            min_r_frac=MIN_R_FRAC, max_r_frac=MAX_R_FRAC, center_tol=CENTER_TOL,
            area_min_ratio=CROP_AREA_MIN_RATIO, area_max_ratio=CROP_AREA_MAX_RATIO, center_fallback_r_frac=CENTER_FALLBACK_R_FRAC,
        )

        stats['source_kind'][source_kind] += 1
        if crop_status not in stats['crop_status']:
            stats['crop_status'][crop_status] = 0
        stats['crop_status'][crop_status] += 1

        rel = img_path.relative_to(raw_root).with_suffix('')
        sample_id = unique_id(sanitize_id(str(rel).replace('\\', '__').replace('/', '__')), used_ids)

        for a_idx in range(int(AUG_MULTIPLIER)):
            if a_idx == 0:
                aug_img, aug_inst = crop_img, crop_inst
            else:
                aug_img, aug_inst = augment_pair(crop_img, crop_inst, rng)
            aug_img, aug_inst = force_size_pair(aug_img, aug_inst, TARGET_SIZE)
            out_id = f'{sample_id}__a{a_idx:02d}'
            write_image_rgb(out_images / f'{out_id}.jpg', aug_img)
            write_png_mask(out_instances / f'{out_id}.png', aug_inst.astype(np.uint16))
            stats['saved_with_augs'] += 1

        stats['processed_base_samples'] += 1

    val = validation_stats(out_images, out_instances)
    report = {
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'source_root': str(raw_root.resolve()),
        'output_root': str(out_root.resolve()),
        'target_format': 'images/* + instances/*.png (0=background, 1..N=instance ids)',
        'augmentation_multiplier': int(AUG_MULTIPLIER),
        'include_unlabeled_as_empty': bool(INCLUDE_UNLABELED_AS_EMPTY),
        'preprocess': {'final_resize_size': int(TARGET_SIZE), 'jpeg_quality_min': 95},
        'crop_config': {
            'use_mask_guided_crop': bool(USE_MASK_GUIDED_CROP),
            'mask_outside': bool(MASK_OUTSIDE_CIRCLE),
            'pad': float(PAD_FRAC),
            'min_r_frac': float(MIN_R_FRAC),
            'max_r_frac': float(MAX_R_FRAC),
            'center_tol': float(CENTER_TOL),
            'area_min_ratio': float(CROP_AREA_MIN_RATIO),
            'area_max_ratio': float(CROP_AREA_MAX_RATIO),
            'center_fallback_r_frac': float(CENTER_FALLBACK_R_FRAC),
        },
        'stats': stats,
        'skipped_examples': skipped_examples,
        'validation': val,
    }
    (out_root / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    return report

def merge_into_pool(out_root: Path, pool_root: Path, pool_prefix: str, dataset_key: str) -> dict:
    src_images, src_instances = out_root / 'images', out_root / 'instances'
    dst_images, dst_instances = pool_root / 'images', pool_root / 'instances'
    ensure_dir(dst_images); ensure_dir(dst_instances)
    copied = 0; skipped_missing_pairs = 0
    for img_path in sorted(src_images.iterdir()):
        if not img_path.is_file():
            continue
        inst_path = src_instances / f'{img_path.stem}.png'
        if not inst_path.exists():
            skipped_missing_pairs += 1
            continue
        out_stem = f'{pool_prefix}__{img_path.stem}'
        shutil.copy2(img_path, dst_images / f'{out_stem}{img_path.suffix.lower()}')
        shutil.copy2(inst_path, dst_instances / f'{out_stem}.png')
        copied += 1

    val = validation_stats(dst_images, dst_instances)
    report_path = pool_root / 'report.json'
    pool_report = json.loads(report_path.read_text(encoding='utf-8')) if report_path.exists() else {}
    sources = pool_report.get('sources', [])
    if not isinstance(sources, list): sources = []

    ds_count = len([p for p in dst_images.iterdir() if p.is_file() and p.stem.startswith(pool_prefix + '__')])
    updated = False
    for s in sources:
        if isinstance(s, dict) and s.get('dataset') == dataset_key:
            s['added'] = int(ds_count); s['skipped'] = 0; updated = True; break
    if not updated:
        sources.append({'dataset': dataset_key, 'added': int(ds_count), 'skipped': 0})

    total_added = int(sum(int(s.get('added', 0)) for s in sources if isinstance(s, dict)))
    total_skipped = int(sum(int(s.get('skipped', 0)) for s in sources if isinstance(s, dict)))
    pool_report['generated_at_utc'] = datetime.now(timezone.utc).isoformat()
    pool_report['output_root'] = str(pool_root.resolve())
    pool_report['sources'] = sources
    pool_report['totals'] = {'added_all': total_added, 'skipped_all': total_skipped}
    pool_report['validation'] = val
    report_path.write_text(json.dumps(pool_report, ensure_ascii=False, indent=2), encoding='utf-8')

    return {'copied_to_pool': copied, 'skipped_missing_pairs': skipped_missing_pairs, 'dataset_pairs_total_in_pool': int(ds_count), 'pool_validation': val}


In [ ]:
convert_report = build_dataset(RAW_ROOT, OUT_ROOT)
pool_report = merge_into_pool(OUT_ROOT, POOL_ROOT, POOL_PREFIX, REPORT_DATASET_KEY)
print(json.dumps({'dataset': convert_report, 'trainable_pool': pool_report}, ensure_ascii=False, indent=2))


In [ ]:
proc_images = sorted((OUT_ROOT / 'images').glob('*'))
if len(proc_images) == 0:
    raise RuntimeError('No processed images found. Run previous cell first.')
pick = proc_images[np.random.randint(0, len(proc_images))]
inst_path = OUT_ROOT / 'instances' / f'{pick.stem}.png'
img_bgr = read_image_any(pick, cv2.IMREAD_COLOR)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
inst = read_image_any(inst_path, cv2.IMREAD_UNCHANGED)
ov = overlay_boundaries(img_rgb, inst)
plt.figure(figsize=(14, 5))
plt.subplot(1, 3, 1); plt.title('Image'); plt.imshow(img_rgb); plt.axis('off')
plt.subplot(1, 3, 2); plt.title('Instances'); plt.imshow(inst); plt.axis('off')
plt.subplot(1, 3, 3); plt.title('Overlay boundaries'); plt.imshow(ov); plt.axis('off')
plt.tight_layout()
